# NLP Part 2

Dalam workbook ini anda akan belajar bagaimana cara membangun sentiment analysis dengan menggunakan metode unsupervised machine learning. Berikut merupakan langkah-langkah dalam melakukan teks analitik
1. Preprocessing Data
2. Modelling
3. Evaluasi Model

## Connect Gdrive to Colab
Sebelum memulai, pastikan bahwa google colab anda sudah tersambung dengan google drive anda.


In [1]:
# Mengakses google drive ke dalam google colaboratory
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


## Import Package

In [2]:
!pip3 install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 6.2 MB/s eta 0:00:00


In [3]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 27.1 MB/s eta 0:00:00


In [4]:
# Import Package

import os
import re
import multiprocessing
import pandas as pd
import numpy as np
from time import time
from unidecode import unidecode

import nltk
from nltk.util import ngrams

from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
from IPython.display import display

In [5]:
# Download Corpus
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
# Mendefinisikan path dan cek keberadaan data
path = '/content/gdrive/MyDrive/Data Notebook/SDS/NLP/'


os.listdir(path)

['.ipynb_checkpoints',
 'news_dataset.csv',
 'twitter_training.csv',
 'movierating.csv',
 'spam.csv',
 'reviews.csv',
 'Amazon_Reviews.csv',
 'Airline_review.csv',
 'all-data.csv',
 'result_word2vec.model']

## Load Data
Data yang akan anda gunakan adalah data yang diambil dari twitter, dalam data ini, anda hanya memiliki data text saja. Selanjutnya anda ingin mencari tahu bagaimana sentiment dari user pengguna twitter.

In [9]:
# Load dataset dari file CSV yang ada di path yang ditentukan.
# Menggunakan 'os.path.join' untuk menggabungkan path direktori dan nama file.
# Menggunakan 'encoding="ISO-8859-1"' karena dataset mungkin berisi karakter khusus.
# 'header=None' menunjukkan bahwa file CSV tidak memiliki baris header.
df_ = pd.read_csv(os.path.join(path, 'Amazon_Reviews.csv'), encoding = "ISO-8859-1", header=None)
# Menampilkan 5 baris pertama dari DataFrame untuk melihat struktur data awal.
df_.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,Name,Place,Job_type,Department,Date,Overall_rating,work_life_balance,skill_development,salary_and_benefits,job_security,career_growth,work_satisfaction,Likes,Dislikes
1,Software Development Engineer,Hyderabad/Secunderabad,Full Time,Software Development Department,8 Aug 2023,2.0,1.0,2.0,5.0,1.0,3.0,3.0,The office is great and you have best in indus...,Worst work life balance.\nThe managers have to...
2,Dls Case Specialist II HR Assistant,Hyderabad/Secunderabad,Full Time,HR Operations Department,8 Aug 2023,1.0,5.0,1.0,3.0,1.0,2.0,1.0,Good work life balance and team building exerc...,No job security. They will cut you out any-day...
3,Customer Support Associate (working remotely),NaN,Full Time,Customer Success Department,8 Aug 2023,1.0,2.0,1.0,2.0,1.0,2.0,3.0,"good about this company, provides virtual roles","this company is really bad , no job security ,..."
4,Sds Associate (working remotely),NaN,Full Time,Non Voice Department,7 Aug 2023,2.0,3.0,2.0,2.0,2.0,2.0,2.0,Leaves are there but more than policy it will ...,To be honest there are Many consumer team but ...


In [10]:
# Merubah nama kolom dalam dataframe
# Mengambil kolom terakhir dari DataFrame asli (df_) yang berisi teks 'Dislikes'.
df = df_.iloc[ :, -1:]
# Memberi nama kolom yang baru saja diambil menjadi 'text'.
df.columns = ['text']
# Menampilkan 5 baris pertama dari DataFrame yang sudah diubah kolomnya untuk verifikasi.
df.head()

,text
0,Dislikes
1,Worst work life balance.\nThe managers have to...
2,No job security. They will cut you out any-day...
3,"this company is really bad , no job security ,..."
4,To be honest there are Many consumer team but ...


Sekarang anda telah mengubah nama kolom dan memiliki text data yang akan anda analisis sentimentnya. Namun sebelum itu akan dilakukan cleansing terlebih dahulu untuk memastikan tidak ada data yang duplikasi atau missing.

## Preprocessing
Mencari informasi dari data, baik jumlah baris dan kolom, maupun tipe dari data yang dimiliki.

In [11]:
# Melihat informasi data dari DataFrame df.
# Fungsi `.info()` memberikan ringkasan DataFrame, termasuk tipe data setiap kolom,
# jumlah entri non-null (untuk mengecek missing value), dan penggunaan memori.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9991 entries, 0 to 9990
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    9991 non-null   object
dtypes: object(1)
memory usage: 78.2+ KB


Data yang akan digunakan terdapat 1 kolom dan juga memiliki 73824 baris data. Selanjutnya akan dicari tahu apakah ada data yang missing atau tidak.

In [12]:
# Cek missing value
# Melakukan pengecekan jumlah nilai yang hilang (missing values) untuk setiap kolom dalam DataFrame df.
# `.isnull()` akan mengembalikan DataFrame boolean yang menunjukkan True di mana ada nilai null dan False sebaliknya.
# `.sum()` kemudian akan menghitung jumlah True (yaitu, jumlah missing values) per kolom.
df.isnull().sum()

,0
text,0


Dari data yang ada, tidak ditemukan ada missing value, selanjutnya akan di cari tahu apakah ada data yang duplicate atau tidak.

In [13]:
# Cek data yang duplicated
# Memfilter DataFrame `df` untuk menampilkan baris-baris yang merupakan duplikasi.
# `df.duplicated()` mengembalikan Series boolean, True untuk baris yang terduplikasi (setelah kemunculan pertama).
df[df.duplicated()]

,text
11,Worst work life balance.\nThe managers have to...
12,No job security. They will cut you out any-day...
13,"this company is really bad , no job security ,..."
14,To be honest there are Many consumer team but ...
15,Management layer has lot of redundancies.
...,...
9986,Less visibility for next level and promotions ...
9987,Amazon has adopted giving responsibility of Wo...
9988,It was a terrible experience working with amaz...
9989,Logistics supply management application operat...


Dari pengecekan duplicated data, ditemukan terdapat beberapa data yang terduplikasi, selanjutnya akan dilakukan cleaning untuk data yang terduplikasi.

In [15]:
# Menghapus data yang terduplikasi dari DataFrame df.
# `drop_duplicates()` akan menghapus baris-baris yang duplikat, hanya menyisakan satu instans dari setiap baris unik.
# `reset_index(drop=True)` akan mengatur ulang indeks DataFrame setelah penghapusan baris, dan `drop=True` memastikan indeks lama tidak ditambahkan sebagai kolom baru.
df = df.drop_duplicates().reset_index(drop=True)

In [16]:
# Menampilkan dimensi (jumlah baris dan kolom) dari DataFrame df setelah penghapusan duplikasi.
df.shape

(11, 1)

Setelah memastikan bahwa data yang anda miliki sudah tidak terdapat missing value maupun duplikasi data, selanjutnya anda perlu melakukan cleaning pada tiap value dalam text data yang anda miliki. Konsep dari cleaning yang dilakukan sama dengan materi yang anda pelajari sebelumnya, namun terdapat beberapa step yang tidak akan anda gunakan. Berikut proses cleaning yang akan anda lakukan.

In [17]:
# Mendefinisikan 'stops' sebagai set (kumpulan) kata-kata umum (stopwords) dalam bahasa Inggris.
# Kata-kata ini biasanya diabaikan dalam analisis teks karena tidak banyak berkontribusi pada makna.
stops = set(nltk.corpus.stopwords.words("english"))

In [18]:
# Data preprocessing

# Pola regex untuk menghilangkan tag HTML (misal: <br>, <p>)
html_tag = re.compile(r'<.*?>')
# Pola regex untuk menghilangkan link HTTP (misal: https://example.com)
http_link = re.compile(r'https://\S+')
# Pola regex untuk menghilangkan link WWW (misal: www.example.com)
www_link = re.compile(r'www\.\S+')

# Pola regex untuk menghilangkan mention akun user (misal: @username)
user_name = re.compile(r'\@[a-z0-9]+')

# Pola regex untuk menghilangkan tanda baca yang tidak diperlukan
# [^\w\s] berarti mencocokkan karakter apa pun yang BUKAN huruf, angka, atau spasi
punctuation = re.compile(r'[^\w\s]')

# Fungsi untuk memproses pembersihan teks data
def data_cleaning(text, stopwords = False):
  # Mengubah teks menjadi format Unicode standar, menghilangkan karakter non-ASCII
  text = unidecode(text)

  # Mengubah semua teks menjadi huruf kecil (lowercase)
  text = text.lower()

  # Menghilangkan tag HTML dari teks
  text = re.sub(html_tag, r'', text)

  # Menghilangkan URL yang dimulai dengan http:// atau https://
  text = re.sub(http_link, r'', text)
  # Menghilangkan URL yang dimulai dengan www.
  text = re.sub(www_link, r'', text)

  # Menghilangkan nama pengguna (username) yang diawali dengan '@'
  text = re.sub(user_name, r'', text)

  # Menghilangkan tanda baca dari teks
  text = re.sub(punctuation, r'', text)

  # Memecah teks menjadi daftar kata-kata (tokenize)
  text = text.split()

  # Jika parameter 'stopwords' adalah True, hilangkan stopwords dari daftar kata
  if stopwords:
    text = [w for w in text if not w in stops]
  # Mengembalikan daftar kata yang sudah bersih
  return text

In [19]:
# Menerapkan cleaning pada dataset
# Membuat kolom baru 'amazon_clean' dalam DataFrame df.
# Untuk setiap baris di kolom 'text', fungsi `data_cleaning` dipanggil dengan `stopwords=True`
# untuk membersihkan teks dan menghilangkan stopwords.
df['amazon_clean'] = df['text'].apply(lambda x: data_cleaning(x, stopwords=True))
# Menampilkan 5 baris pertama dari DataFrame yang sudah memiliki kolom 'amazon_clean' untuk verifikasi.
df.head()

,text,amazon_clean
0,Dislikes,[dislikes]
1,Worst work life balance.\nThe managers have to...,"[worst, work, life, balance, managers, ask, wo..."
2,No job security. They will cut you out any-day...,"[job, security, cut, anyday, anytime, based, b..."
3,"this company is really bad , no job security ,...","[company, really, bad, job, security, hr, gues..."
4,To be honest there are Many consumer team but ...,"[honest, many, consumer, team, team, working, ..."


In [20]:
# Menampilkan dimensi (jumlah baris dan kolom) dari DataFrame data_clean.
df.shape

(11, 2)

In [21]:
# Memilih data yang minimal mempunyai 2 kata didalamnya
data_clean = df[df.amazon_clean.str.len() > 1].reset_index(drop=True)
data_clean.shape

(10, 2)

Terdapat perbedaan jumlah baris sebelum dan sesudah filtering dengan menggunakan jumlah kata dalam dataset.

In [22]:
# Mengecek kondisi data saat ini
data_clean.head()

,text,amazon_clean
0,Worst work life balance.\nThe managers have to...,"[worst, work, life, balance, managers, ask, wo..."
1,No job security. They will cut you out any-day...,"[job, security, cut, anyday, anytime, based, b..."
2,"this company is really bad , no job security ,...","[company, really, bad, job, security, hr, gues..."
3,To be honest there are Many consumer team but ...,"[honest, many, consumer, team, team, working, ..."
4,Management layer has lot of redundancies.,"[management, layer, lot, redundancies]"


## N-Gram
Konsep dari N-gram adalah mentoken kata berdasarkan n kata. N-gram model dapat membantu untuk mengenali beberapa konteks kata yang tidak bisa terpisahkan. Sebelum anda melanjutkan step berikutnya, akan dikenalkan terlebih dahulu mengenai konsep n-gram.

In [23]:
# Konsep n-gram
# Akan ditunjukkan dengan 1 data teratas
unigram = []
bigram = []
trigram = []

# Mengambil kata-kata dari baris pertama 'amazon_clean' dalam DataFrame data_clean
# Kemudian melakukan iterasi untuk membuat unigram, bigram, dan trigram.
for words in data_clean["amazon_clean"][:1]:
  # Membuat bigram (pasangan dua kata berurutan) dari daftar kata-kata
  list_bigram = list(ngrams(words, 2))
  # Membuat trigram (pasangan tiga kata berurutan) dari daftar kata-kata
  list_trigram = list(ngrams(words, 3))
  # Menambahkan setiap kata ke dalam list unigram
  for word in words:
    unigram.append(word)
  # Menambahkan setiap bigram ke dalam list bigram
  for word in list_bigram:
    bigram.append(word)
  # Menambahkan setiap trigram ke dalam list trigram
  for word in list_trigram:
    trigram.append(word)

# Menampilkan kalimat asli yang digunakan sebagai contoh
print("Kalimat : ", data_clean["amazon_clean"][:1])
# Menampilkan unigram (setiap kata tunggal)
print("Unigram : ", unigram)
# Menampilkan bigram (pasangan dua kata)
print("Bigram : ", bigram)
# Menampilkan trigram (pasangan tiga kata)
print("Trigram : ", trigram)

Kalimat :  0    [worst, work, life, balance, managers, ask, wo...
Name: amazon_clean, dtype: object
Unigram :  ['worst', 'work', 'life', 'balance', 'managers', 'ask', 'work', 'least', '10', 'hrs', 'day', 'work', '12', 'performance', 'measured', 'team', 'everyone', 'working', '12', 'hours', 'work', '8', 'always', 'bottom', 'performer', 'promotions', 'project', 'based', 'yourprojects', 'gets', 'halted', 'reason', 'promotion']
Bigram :  [('worst', 'work'), ('work', 'life'), ('life', 'balance'), ('balance', 'managers'), ('managers', 'ask'), ('ask', 'work'), ('work', 'least'), ('least', '10'), ('10', 'hrs'), ('hrs', 'day'), ('day', 'work'), ('work', '12'), ('12', 'performance'), ('performance', 'measured'), ('measured', 'team'), ('team', 'everyone'), ('everyone', 'working'), ('working', '12'), ('12', 'hours'), ('hours', 'work'), ('work', '8'), ('8', 'always'), ('always', 'bottom'), ('bottom', 'performer'), ('performer', 'promotions'), ('promotions', 'project'), ('project', 'based'), ('based

Dari output diatas anda dapat mengetahui bahwa terdapat perbedaan output dalam pemisahan pengelompokan kata. Model N-gram banyak digunakan untuk mencari prediksi kata berikutnya yang akan muncul ketika terdapat history text.

In [ ]:
help(Phrases)

Help on class Phrases in module gensim.models.phrases:

class Phrases(_PhrasesTransformation)
 |  Phrases(sentences=None, min_count=5, threshold=10.0, max_vocab_size=40000000, delimiter='_', progress_per=10000, scoring='default', connector_words=frozenset())
 |
 |  Detect phrases based on collocation counts.
 |
 |  Method resolution order:
 |      Phrases
 |      _PhrasesTransformation
 |      gensim.interfaces.TransformationABC
 |      gensim.utils.SaveLoad
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, sentences=None, min_count=5, threshold=10.0, max_vocab_size=40000000, delimiter='_', progress_per=10000, scoring='default', connector_words=frozenset())
 |      Parameters
 |      ----------
 |      sentences : iterable of list of str, optional
 |          The `sentences` iterable can be simply a list, but for larger corpora, consider a generator that streams
 |          the sentences directly from disk/network, See :class:`~gensim.models.word2vec.BrownCorpu

In [24]:
# Menggunakan model Phrases untuk mendeteksi frasa bigram dari dataset yang telah dibersihkan.
# 'sent' adalah daftar kalimat (setiap kalimat adalah daftar kata).
sent = [row for row in data_clean.amazon_clean]

# Membuat objek Phrases. min_count=3 berarti hanya frasa yang muncul minimal 3 kali akan dipertimbangkan.
# progress_per=50000 menampilkan kemajuan setiap 50.000 kalimat diproses.
phrases = Phrases(sent, min_count=3, progress_per=50000)

# Membuat objek Phraser dari Phrases. Phraser lebih cepat dan efisien untuk digunakan setelah model Phrases dilatih.
bigram = Phraser(phrases)

# Mengaplikasikan model bigram ke setiap kalimat di 'sent' untuk menggabungkan kata-kata yang membentuk frasa.
sentences = bigram[sent]

# Menampilkan kalimat pertama setelah bigram diaplikasikan untuk melihat hasilnya.
sentences[1]

['job',
 'security',
 'cut',
 'anyday',
 'anytime',
 'based',
 'business',
 'requirements',
 'performance',
 'wonat',
 'even',
 'gonna',
 'matter']

Syntax mengunakan konsep bigram dimana ketika didalam model dikenali kata yang merupakan 1 kesatuan, maka secara otomatis akan tersambung kedalam 1 kata. Berikut merupakan contohnya.

In [25]:
example = 'commonly known as the united states'
example = example.split()
bigram[example]

['commonly', 'known', 'as', 'the', 'united', 'states']

## POS TAGGING

In [26]:
nltk.download('words')
nltk.download('averaged_perceptron_tagger')
nltk.download('tagsets')

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package tagsets to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets.zip.


True

In [27]:
# Akan di contohkan dalam 1 kalimat
pos_tag = []

text = ["European authorities fined Google a record $5.1 billion on Wednesday for abusing its power in the mobile phone market and ordered the company to alter its practices."]

# Download resource NLTK yang mungkin belum ada (jika sudah ada, ini tidak akan melakukan apa-apa)
nltk.download('averaged_perceptron_tagger_eng')

# Melakukan iterasi untuk setiap kalimat dalam list 'text' (dalam kasus ini hanya ada satu kalimat)
for i in text:
  # Memecah kalimat menjadi kata-kata terpisah
  words = i.split()
  # Melakukan POS Tagging pada setiap kata untuk menentukan jenis katanya (misalnya, kata benda, kata kerja, kata sifat)
  words = nltk.pos_tag(words)
  # Menambahkan setiap pasangan kata dan tag-nya ke dalam list pos_tag
  for j in words:
    pos_tag.append(j)

# Menampilkan hasil POS Tagging
pos_tag

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


[('European', 'JJ'),
 ('authorities', 'NNS'),
 ('fined', 'VBD'),
 ('Google', 'NNP'),
 ('a', 'DT'),
 ('record', 'NN'),
 ('$5.1', 'NN'),
 ('billion', 'CD'),
 ('on', 'IN'),
 ('Wednesday', 'NNP'),
 ('for', 'IN'),
 ('abusing', 'VBG'),
 ('its', 'PRP$'),
 ('power', 'NN'),
 ('in', 'IN'),
 ('the', 'DT'),
 ('mobile', 'JJ'),
 ('phone', 'NN'),
 ('market', 'NN'),
 ('and', 'CC'),
 ('ordered', 'VBD'),
 ('the', 'DT'),
 ('company', 'NN'),
 ('to', 'TO'),
 ('alter', 'VB'),
 ('its', 'PRP$'),
 ('practices.', 'NN')]

Keterangan seperti NN, JJ, NNS dan lain-lain menunjukkan susunan dari kata tersebut. Untuk lebih detailnya anda bisa mengkonfirmasi makna dari pos tagging tersebut dengan cara

In [28]:
nltk.download('tagsets_json')
nltk.help.upenn_tagset('JJ')

JJ: adjective or numeral, ordinal
    third ill-mannered pre-war regrettable oiled calamitous first separable
    ectoplasmic battery-powered participatory fourth still-to-be-named
    multilingual multi-disciplinary ...


[nltk_data] Downloading package tagsets_json to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets_json.zip.


## NER (Named Entity Recognition)

In [29]:
nltk.download('maxent_ne_chunker')

[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.


True

In [30]:
nltk.download('maxent_ne_chunker_tab')
# Menggunakan fungsi `nltk.ne_chunk` untuk melakukan Named Entity Recognition (NER) pada hasil POS tagging sebelumnya.
# `pos_tag` adalah daftar kata yang sudah diberi label POS.
# `binary=True` berarti NER akan mengidentifikasi entitas sebagai entitas generik (misalnya, 'NE' untuk 'Named Entity') tanpa mengklasifikasikan secara spesifik (misalnya, 'PERSON', 'ORGANIZATION').
chunks = nltk.ne_chunk(pos_tag, binary = True)
# Melakukan iterasi pada setiap 'chunk' (potongan) yang dihasilkan oleh NER.
# Chunk bisa berupa entitas bernama atau kata biasa yang tidak teridentifikasi sebagai entitas.
for chunk in chunks:
  # Mencetak setiap chunk. Untuk entitas bernama, akan muncul sebagai struktur pohon. Untuk kata biasa, akan muncul sebagai tuple (kata, tag_POS).
  print(chunk)

[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.


(NE European/JJ)
('authorities', 'NNS')
('fined', 'VBD')
('Google', 'NNP')
('a', 'DT')
('record', 'NN')
('$5.1', 'NN')
('billion', 'CD')
('on', 'IN')
('Wednesday', 'NNP')
('for', 'IN')
('abusing', 'VBG')
('its', 'PRP$')
('power', 'NN')
('in', 'IN')
('the', 'DT')
('mobile', 'JJ')
('phone', 'NN')
('market', 'NN')
('and', 'CC')
('ordered', 'VBD')
('the', 'DT')
('company', 'NN')
('to', 'TO')
('alter', 'VB')
('its', 'PRP$')
('practices.', 'NN')


In [31]:
entities = []
labels = []
# Melakukan iterasi pada setiap 'chunk' yang dihasilkan oleh NER.
for chunk in chunks:
  # Memeriksa apakah chunk memiliki atribut 'label', yang menandakan itu adalah Named Entity.
  if hasattr(chunk, 'label'):
    # Jika ya, gabungkan kata-kata dalam chunk menjadi satu string dan tambahkan ke daftar 'entities'.
    entities.append(" ".join(c[0] for c in chunk))
    # Tambahkan label dari chunk tersebut ke daftar 'labels'.
    labels.append(chunk.label())

# Menggabungkan entitas dan label menjadi pasangan unik (untuk menghindari duplikasi) dan mengubahnya menjadi list.
entities_labels = list(set(zip(entities, labels)))
# Membuat DataFrame dari pasangan entitas dan label.
entities_df = pd.DataFrame(entities_labels)
# Mengubah nama kolom DataFrame menjadi 'entities' dan 'label'.
entities_df.columns = ["entities", "label"]
# Menampilkan DataFrame hasil.
entities_df

,entities,label
0,European,NE


## Word2Vec
Word2vec adalah salah satu konsep embedding yang akan mengubah kata kedalam bentuk vector angka. Bentuk numerik dari tiap kata ini dapat digunakan untuk menentukan seberapa dekat dan mirip penggolongan kata-kata tersebut.

In [32]:
# Set Parameter
# min_count: Kata-kata dengan frekuensi lebih rendah dari ini akan diabaikan.
min_count = 3
# window: Ukuran jendela maksimum untuk kata-kata di sekitar kata target.
window = 4
# size: Dimensi vektor (ukuran embedding) untuk setiap kata.
size = 250
# sample: Batas ambang untuk mensampling kata-kata yang sering muncul secara acak.
sample = 1e-5
# negative: Jumlah 'kata negatif' yang digunakan untuk negative sampling.
negative = 20
# workers: Jumlah core CPU yang akan digunakan untuk melatih model.
workers = multiprocessing.cpu_count()-1

min_count digunakan untuk menghapus huruf yang tidak biasa atau typo seperti kemunculan huruf s sebanyak 3 kali berturut-turut.<br>
window digunakan untuk belajar memprediksi kata yang diberikan dari 4 kata ke kiri, dan hingga 4 kata ke kanan.<br>
size adalah size hidden layer.<br>
sample adalah probalititas kata yang sering muncul.<br>
negative adalah jumlah kata negatif.

In [33]:
# Membuat objek Word2Vec dengan parameter yang telah ditentukan.
# min_count: Kata-kata dengan frekuensi lebih rendah dari ini akan diabaikan (disini 3).
# window: Ukuran jendela maksimum untuk kata-kata di sekitar kata target (disini 4).
# vector_size: Dimensi vektor (ukuran embedding) untuk setiap kata (disini 250).
# sample: Batas ambang untuk mensampling kata-kata yang sering muncul secara acak (disini 1e-5).
# negative: Jumlah 'kata negatif' yang digunakan untuk negative sampling (disini 20).
# workers: Jumlah core CPU yang akan digunakan untuk melatih model (jumlah core CPU - 1).
w2v_model = Word2Vec(min_count=min_count,
                     window=window,
                     vector_size=size,
                     sample=sample,
                     negative=negative,
                     workers=workers)

start = time()

# Membangun kosakata (vocabulary) dari kumpulan kalimat yang sudah diproses.
# progress_per=50000 akan menampilkan kemajuan setiap 50.000 kalimat diproses.
w2v_model.build_vocab(sentences, progress_per=50000)

# Menampilkan waktu yang dibutuhkan untuk membangun kosakata.
print('Waktu untuk membangun vocab : {} menit'.format(round((time() - start) / 60, 2)))

Waktu untuk membangun vocab : 0.0 menit


In [34]:
# Membangun model word2vec
start = time()

w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

print('Waktu untuk train model : {} menit'.format(round((time() - start) / 60, 2)))

w2v_model.init_sims(replace=True)

/tmp/ipython-input-3463618051.py:8: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  w2v_model.init_sims(replace=True)


Waktu untuk train model : 0.0 menit


In [36]:
# Menyimpan model Word2Vec yang telah dilatih.
# Mendefinisikan nama file untuk model yang akan disimpan.
model_name = "result_word2vec.model"
# Menggunakan metode .save() dari objek w2v_model untuk menyimpan model ke disk.
# os.path.join(path, model_name) digunakan untuk menggabungkan jalur direktori (path) dengan nama file (model_name)
# sehingga model tersimpan di lokasi yang benar di Google Drive.
w2v_model.save(os.path.join(path, model_name))

In [37]:
# replace bigram
# Membuat salinan DataFrame 'data_clean' ke 'data_final' agar tidak mengubah data asli.
data_final = data_clean.copy()
# Menyimpan versi teks asli yang sudah dibersihkan (sebelum bigram diaplikasikan) ke kolom 'old_text'.
data_final['old_text'] = data_final.amazon_clean

# Menggabungkan kembali token-token kata dalam kolom 'old_text' menjadi satu string dengan spasi sebagai pemisah.
data_final.old_text = data_final.old_text.str.join(' ')
# Menerapkan model bigram ke setiap kalimat di kolom 'amazon_clean' untuk menggabungkan frasa bigram.
# Setelah bigram diterapkan, token-token digabungkan kembali menjadi satu string dengan spasi.
data_final.amazon_clean = data_final.amazon_clean.apply(lambda x: ' '.join(bigram[x]))
# Menampilkan 5 baris pertama dari DataFrame 'data_final' untuk melihat hasilnya.
data_final.head()

,text,amazon_clean,old_text
0,Worst work life balance.\nThe managers have to...,worst work life balance managers ask work leas...,worst work life balance managers ask work leas...
1,No job security. They will cut you out any-day...,job security cut anyday anytime based business...,job security cut anyday anytime based business...
2,"this company is really bad , no job security ,...",company really bad job security hr guess well ...,company really bad job security hr guess well ...
3,To be honest there are Many consumer team but ...,honest many consumer team team working worst l...,honest many consumer team team working worst l...
4,Management layer has lot of redundancies.,management layer lot redundancies,management layer lot redundancies


## Modelling
Untuk mencari label dari text yang ada, anda akan menggunakan konsep clustering text. Akan dicontohkan untuk membagi text kedalam 2 cluster, positif dan negatif dengan menggunakan k-means.

In [38]:
# Membangun model dengan 2 cluster
word_vectors = w2v_model.wv
model = KMeans(n_clusters=2, max_iter=1000, random_state=True, n_init=50).fit(X=word_vectors.vectors.astype('double'))

In [40]:
word_vectors.similar_by_vector(model.cluster_centers_[1], topn=10, restrict_vocab=None)

[('skills', 0.5379988551139832),
 ('security', 0.5263742208480835),
 ('trainer', 0.48383083939552307),
 ('job', 0.4141477644443512),
 ('team', 0.4016640782356262),
 ('comm', -0.018205460160970688),
 ('working', -0.03502655774354935),
 ('amazon', -0.07924504578113556),
 ('work', -0.13531827926635742)]

In [42]:
word_vectors.similar_by_vector(model.cluster_centers_[0], topn=10, restrict_vocab=None)

[('amazon', 0.5366182923316956),
 ('comm', 0.5181772708892822),
 ('working', 0.49240660667419434),
 ('work', 0.4736180901527405),
 ('trainer', 0.030449554324150085),
 ('team', -0.03172127157449722),
 ('skills', -0.08665234595537186),
 ('job', -0.09319362044334412),
 ('security', -0.13215728104114532)]

Dari output diatas, maka anda dapat melihat bahwa kata-kata yang muncul dengan cluster center 1 adalah kata-kata positif, oleh karena itu untuk cluster 1 maka didapat sentiment positif dan 0 untuk negatif.

In [43]:
# Mendefinisikan tiap cluster
positive_cluster_index = 1
positive_cluster = model.cluster_centers_[positive_cluster_index]
negative_cluster = model.cluster_centers_[1-positive_cluster_index]


In [45]:
# words = pd.DataFrame(word_vectors.vocab.keys())

# Membuat DataFrame baru dari kata-kata unik yang ada di vocabulary model Word2Vec.
# word_vectors.index_to_key berisi daftar kata-kata unik yang telah di-embedding.
words = pd.DataFrame(word_vectors.index_to_key)
# Mengubah nama kolom pertama menjadi 'words' agar lebih deskriptif.
words.columns = ['words']

# Membuat kolom baru 'vectors' yang berisi vektor numerik (embedding) untuk setiap kata.
# word_vectors[f'{x}'] mengambil vektor untuk kata 'x' dari model Word2Vec.
words['vectors'] = words.words.apply(lambda x: word_vectors[f'{x}'])

In [46]:
# menambah kolom cluster untuk prediksi sentiment tiap kata
# Membuat kolom 'cluster' baru di DataFrame 'words'.
# Setiap vektor kata di kolom 'vectors' akan diprediksi cluster-nya menggunakan model KMeans yang telah dilatih.
words['cluster'] = words.vectors.apply(lambda x: model.predict([np.array(x)]))
# Karena `model.predict` mengembalikan array, kita ambil elemen pertama (hasil prediksi cluster ID).
words.cluster = words.cluster.apply(lambda x: x[0])
# Menampilkan 5 baris pertama dari DataFrame 'words' untuk melihat kolom 'cluster' yang baru ditambahkan.
words.head()

,words,vectors,cluster
0,team,"[-0.0059647886, 0.0026299728, 0.056767732, 0.1...",1
1,work,"[0.015663661, -0.028841611, -0.07710454, -0.08...",0
2,trainer,"[-0.094255455, 0.023005415, -0.009434141, -0.1...",1
3,skills,"[-0.09628914, -0.016054614, 0.10517717, -0.083...",1
4,amazon,"[0.0785661, -0.01737758, 0.0880832, -0.1051640...",0


In [47]:
# Mapping cluster
# Membuat kolom 'cluster_value': jika cluster adalah cluster positif (index 1), maka nilai 1, jika tidak maka -1.
words['cluster_value'] = [1 if i==positive_cluster_index else -1 for i in words.cluster]
# Membuat kolom 'closeness_score': menghitung seberapa dekat vektor kata ke pusat cluster-nya.
# Nilainya adalah 1 dibagi dengan jarak minimum ke salah satu pusat cluster.
words['closeness_score'] = words.apply(lambda x: 1/(model.transform([x.vectors]).min()), axis=1)
# Membuat kolom 'sentiment_coeff': mengalikan 'closeness_score' dengan 'cluster_value'
# untuk mendapatkan koefisien sentimen (positif/negatif) untuk setiap kata.
words['sentiment_coeff'] = words.closeness_score * words.cluster_value
# Menampilkan 5 baris pertama dari DataFrame 'words' dengan kolom-kolom baru.
words.head()

,words,vectors,cluster,cluster_value,closeness_score,sentiment_coeff
0,team,"[-0.0059647886, 0.0026299728, 0.056767732, 0.1...",1,1,1.088677,1.088677
1,work,"[0.015663661, -0.028841611, -0.07710454, -0.08...",0,-1,1.134692,-1.134692
2,trainer,"[-0.094255455, 0.023005415, -0.009434141, -0.1...",1,1,1.142555,1.142555
3,skills,"[-0.09628914, -0.016054614, 0.10517717, -0.083...",1,1,1.182784,1.182784
4,amazon,"[0.0785661, -0.01737758, 0.0880832, -0.1051640...",0,-1,1.184259,-1.184259


Sekarang anda telah memiliki sentiment coeficient dari setiap value dalam dataset, selain itu anda juga telah memiliki text yang sudah di cleaning. Selanjutnya akan dibuat dictionary untuk kata dan sentiment untuk kata tersebut.

In [53]:
# konten dalam distionary ini adalah kata sebagai key dan sentiment sebagai value-nya
sentiment_dict = dict(zip(words.words.values, words.sentiment_coeff.values))

In [55]:
# Membuat objek TfidfVectorizer. Parameter `tokenizer` disetel untuk membagi teks berdasarkan spasi.
# `norm=None` berarti tidak ada normalisasi yang diterapkan pada vektor TF-IDF.
tfidf = TfidfVectorizer(tokenizer=lambda y: y.split(), norm=None)
# Melatih (fit) TF-IDF pada kolom 'amazon_clean' dari data_final.
# Ini akan menghitung IDF (Inverse Document Frequency) untuk setiap kata.
tfidf.fit(data_final.amazon_clean)

# mendapatkan nama-nama feature (yaitu, kata-kata unik) yang dipelajari oleh TF-IDF.
features = pd.Series(tfidf.get_feature_names_out())

# mengubah teks di 'amazon_clean' menjadi representasi matriks TF-IDF.
# Setiap baris mewakili dokumen, dan setiap kolom mewakili kata dengan nilai TF-IDF-nya.
transformed = tfidf.transform(data_final.amazon_clean)

In [56]:
# Function to create a word-to-TFIDF-score dictionary for a *single document*
def create_tfidf_dictionary_for_document(document_sparse_row, features_series):
    # Memastikan bahwa input adalah vektor baris jarang (sparse row vector).
    # Jika dimensinya lebih dari 1 dan bukan hanya satu baris, akan memunculkan error.
    if document_sparse_row.ndim > 1 and document_sparse_row.shape[0] != 1:
        raise ValueError("Expected a single sparse row vector.")

    # Mengkonversi vektor sparse ke format COO (Coordinate format) untuk mendapatkan indeks kolom dan data (skor TF-IDF).
    row_coo = document_sparse_row.tocoo()

    # Memetakan indeks kolom kembali ke nama fitur (kata-kata) yang sesuai.
    words_in_document = features_series.iloc[row_coo.col].values

    # Membuat dictionary: {kata: skor_tfidf}.
    tfidf_dict = dict(zip(words_in_document, row_coo.data))
    return tfidf_dict

# Function to replace words in a text string with their TF-IDF scores
# This function will now be called for each document, passing its index
def get_tfidf_scores_for_text(text_string, doc_index, transformed_matrix, features_series):
    # Mengambil vektor sparse untuk dokumen saat ini (berdasarkan indeks baris).
    document_sparse_row = transformed_matrix[doc_index, :]

    # Membuat dictionary TF-IDF khusus untuk dokumen ini.
    tfidf_dict = create_tfidf_dictionary_for_document(document_sparse_row, features_series)

    # Memisahkan teks yang sudah dibersihkan menjadi kata-kata individual.
    words_in_text = text_string.split()

    # Mendapatkan skor TF-IDF untuk setiap kata. Jika sebuah kata tidak ada dalam dictionary
    # (misalnya, tidak ada dalam vocabulary, atau skor TF-IDF-nya nol), gunakan nilai 0.0.
    return [tfidf_dict.get(word, 0.0) for word in words_in_text]

In [57]:
start = time()

# Apply the function to each row of the DataFrame, passing the index to retrieve the correct TF-IDF vector
# We can use df.index to get the current index, which corresponds to the row in `transformed`
replaced_tfidf_scores = data_final.apply(
    lambda row: get_tfidf_scores_for_text(
        row['amazon_clean'],
        row.name, # Use row.name as the document index, which should match transformed matrix row index
        transformed,
        features
    ),
    axis=1
)

print('Waktu untuk train model : {} menit'.format(round((time() - start) / 60, 2)))

Waktu untuk train model : 0.0 menit


In [58]:
# Membuat function untuk mengubah kata dengan sentiment coefficient

def replace_sentiment_words(word, sentiment_dict):
    # Mencoba untuk mengambil nilai sentimen dari 'sentiment_dict' menggunakan 'word' sebagai kunci.
    try:
        out = sentiment_dict[word]
    # Jika 'word' tidak ditemukan di 'sentiment_dict' (KeyError),
    # maka nilai 'out' diatur menjadi 0.
    except KeyError:
        out = 0
    # Mengembalikan nilai sentimen atau 0 jika kata tidak ditemukan.
    return out

In [59]:
# Mengubah kata dengan sentiment-nya
replaced_closeness_scores = data_final.amazon_clean.apply(lambda x: list(map(lambda y: replace_sentiment_words(y, sentiment_dict), x.split())))
replaced_closeness_scores[:5]

,amazon_clean
0,"[0, -1.1346917820750335, 0, 0, 0, 0, -1.134691..."
1,"[1.096373813310454, 1.1737929732913754, 0, 0, ..."
2,"[0, 0, 0, 1.096373813310454, 1.173792973291375..."
3,"[0, 0, 0, 1.0886770789715945, 1.08867707897159..."
4,"[0, 0, 0, 0]"


In [60]:
# Membuat Dataframe hasil hasil labeling sentiment dan di transformasi
results = pd.DataFrame(data=[replaced_closeness_scores, replaced_tfidf_scores, data_final.amazon_clean]).T

# Mengubah nama kolom
results.columns = ['sentiment_coeff', 'tfidf_scores', 'sentence']

# menggunakan perkalian dot
results['sentiment_rate'] = results.apply(lambda x: np.array(x.loc['sentiment_coeff']).dot(np.array(x.loc['tfidf_scores'])), axis=1)
results['prediction'] = (results.sentiment_rate>0).astype('int8')
results.head()

,sentiment_coeff,tfidf_scores,sentence,sentiment_rate,prediction
0,"[0, -1.1346917820750335, 0, 0, 0, 0, -1.134691...","[2.2992829841302607, 9.197131936521043, 2.7047...",worst work life balance managers ask work leas...,-42.107552,0
1,"[1.096373813310454, 1.1737929732913754, 0, 0, ...","[2.01160091167848, 2.01160091167848, 2.7047480...",job security cut anyday anytime based business...,4.566670,1
2,"[0, 0, 0, 1.096373813310454, 1.173792973291375...","[5.4094961844768505, 2.7047480922384253, 2.704...",company really bad job security hr guess well ...,76.739319,1
3,"[0, 0, 0, 1.0886770789715945, 1.08867707897159...","[2.7047480922384253, 2.7047480922384253, 2.704...",honest many consumer team team working worst l...,0.486011,1
4,"[0, 0, 0, 0]","[2.2992829841302607, 2.7047480922384253, 2.299...",management layer lot redundancies,0.000000,0


In [61]:
# Sederhanakan Output
# Mengambil hanya kolom 'sentence' (kalimat teks) dan 'prediction' (hasil prediksi sentimen)
# dari DataFrame 'results' dan menyimpannya kembali ke 'results'.
results = results[['sentence', 'prediction']]
# Menampilkan 5 baris pertama dari DataFrame 'results' yang sudah disederhanakan.
results.head()

,sentence,prediction
0,worst work life balance managers ask work leas...,0
1,job security cut anyday anytime based business...,1
2,company really bad job security hr guess well ...,1
3,honest many consumer team team working worst l...,1
4,management layer lot redundancies,0
